# Notebook 07 - BIM Data Exchange: JSON and IFC4

This lesson turns a solved Tuba model into exchange artifacts for downstream tools. It covers canonical JSON, IFC-style placement metadata, IFC export with Code_Aster stress/reaction properties, and IFC re-import.

You will do four things:

1. Serialize a model to Tuba JSON and load it back.
2. Attach IFC-style placement metadata.
3. Export IFC4 with Code_Aster-backed engineering properties.
4. Inspect and re-import the IFC model.


## 1. Imports and Setup

In [1]:
import sys
import json
from pathlib import Path
import numpy as np

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model, PlacementAssignment, PlacementFrame
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results
from tuba.external.ifc import IfcExporter, IfcImporter

## 2. JSON Serialization (Tuba's Native AI-Ready Format)

Tuba models are natively serializable to and from dictionary/JSON structures conforming to the Tuba schema.
Let's construct a simple model and look at its JSON representation.

In [2]:
model = Model("BIM_Demo")
model.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850, allowable_stress={20.0: 140e6, 120.0: 125e6})
model.add_pipe_section("DN100", OD=0.1143, WT=0.006)

with model.pipe(section="DN100", material="Steel") as b:
    b.start([0, 0, 0], support="anchor")
    b.run(3.0)
    b.bend(radius=0.2, angle=90.0, plane="XY")
    b.run(2.0)
    b.end(support="anchor")

model.define_load_case("Operating", gravity=True, pressure=1.5e6, temperature=150.0)
print(model)

TubaModel('BIM_Demo', 4 nodes, 3 elements, 2 supports)


### A. IFC-Style Placement Metadata

Tuba keeps native node coordinates in the model-global Cartesian frame, while optional `PlacementFrame` records preserve IFC-style local placement context. Here we attach a product placement frame to the first pipe segment before JSON and IFC export.

The placement origin below uses `first_pipe.n1`, the first endpoint node of the exported pipe segment. This keeps the IFC-style local placement tied to the same endpoint/vector contract used by solver export and visualization.

In [ ]:
first_pipe = next(elem for elem in model.elements if elem.type == "pipe_straight")
frame_id = "pipe_run_frame"
model.placement_frames[frame_id] = PlacementFrame(
    id=frame_id,
    origin=tuple(float(v) for v in model.nodes[first_pipe.n1].coords),
    frame_type="product",
    source="notebook",
    metadata={"description": "Local placement for first exported pipe segment"},
)
model.placement_assignments.append(
    PlacementAssignment(
        target=f"element:{first_pipe.id}",
        frame=f"placement_frame:{frame_id}",
        role="object_placement",
        source="notebook",
    )
)
model.validate()
resolved = model.resolve_placement_frame(f"placement_frame:{frame_id}")
start = model.nodes[first_pipe.n1].coords
end = model.nodes[first_pipe.n2].coords
print(f"Placement frame assigned to {first_pipe.id}")
print(f"First pipe endpoints: n1={first_pipe.n1} {start.tolist()} -> n2={first_pipe.n2} {end.tolist()}")
print(f"  First pipe vector: {(end - start).tolist()}")
print(f"  Origin: {resolved.origin.tolist()}")
print(f"  JSON placement frames: {list(model.placement_frames)}")

### B. Serialize to Python Dict

In [4]:
model_dict = model.to_dict()

# Display first 1500 characters of the formatted JSON dictionary
json_str = json.dumps(model_dict, indent=2)
print(json_str[:1500] + "\n... [truncated] ...")

{
  "meta": {
    "project_name": "BIM_Demo",
    "standard": "ASME_B31.3",
    "version": "4.0.0"
  },
  "materials": {
    "Steel": {
      "E": 200000000000.0,
      "nu": 0.3,
      "rho": 7850,
      "alpha": 1.2e-05,
      "allowable_stress": {
        "20.0": 140000000.0,
        "120.0": 125000000.0
      }
    }
  },
  "sections": {
    "DN100": {
      "type": "pipe",
      "OD": 0.1143,
      "WT": 0.006,
      "corrosion_allowance": 0.0
    }
  },
  "nodes": {
    "N0": [
      0.0,
      0.0,
      0.0
    ],
    "N1": [
      3.0,
      0.0,
      0.0
    ],
    "N2": [
      3.2,
      0.19999999999999998,
      0.0
    ],
    "N3": [
      3.2000000000000006,
      2.2,
      0.0
    ]
  },
  "elements": [
    {
      "id": "pipe_str_0",
      "type": "pipe_straight",
      "n1": "N0",
      "n2": "N1",
      "section": "DN100",
      "material": "Steel",
      "station_start": 0.0,
      "station_end": 3.0
    },
    {
      "id": "pipe_bend_0",
      "type": "pipe_ben

### C. Save to JSON File & Round-Trip Reload

In [5]:
# Save model to json file
json_filepath = "bim_demo_model.json"
model.to_json(json_filepath)
print(f"Model written to {json_filepath}\n")

# Reload the model from the file
reloaded = Model.from_json(json_filepath)
print(f"Reloaded Model name: {reloaded.project_name}")
print(f"  Nodes count: {len(reloaded.nodes)}")
print(f"  Elements count: {len(reloaded.elements)}")
print(f"  Supports count: {len(reloaded.supports)}")

# Assert equality of reloaded properties
assert len(reloaded.nodes) == len(model.nodes)
assert len(reloaded.elements) == len(model.elements)
print("\nAssertion passed: Round-trip JSON reload is 100% loss-less!")

Model written to bim_demo_model.json

Reloaded Model name: BIM_Demo
  Nodes count: 4
  Elements count: 3
  Supports count: 2

Assertion passed: Round-trip JSON reload is 100% loss-less!


## 3. IFC4 Export with Solver Properties

Tuba exports layouts as IFC4 model files. When Code_Aster results are provided, exported products can carry reaction and stress property sets. The next cell loads/runs Code_Aster before writing those properties.


In [6]:
# Load Code_Aster results before exporting solver properties to IFC4.
CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
# VS Code/Jupyter review defaults to committed real Code_Aster artifacts; set True only after the runtime doctor passes.
RUN_CODE_ASTER = False
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "bim_operating"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact

# Export to IFC4 with Code_Aster reactions and stresses.
ifc_filepath = "bim_demo.ifc"
exporter = IfcExporter()
exporter.export_model(model, ifc_filepath, results=results)
print(f"IFC4 file exported to: {ifc_filepath}")
if code_aster_run.ran_solver:
    print("Code_Aster solver executed for this notebook run.")
print(f"Solver result source: {CODE_ASTER_WORK_DIR.resolve()}")

IFC4 file exported to: bim_demo.ifc
Solver result source: D:\Gitprojects\Tuba_v4\notebooks\code_aster_results\bim_operating


## 4. Inspecting Exported IFC Properties with IfcOpenShell

We can load the exported IFC file using `ifcopenshell` and check the metadata and custom properties associated with elements.

In [7]:
try:
    import ifcopenshell
    
    # Open the IFC file
    f = ifcopenshell.open(ifc_filepath)
    
    # Count entities
    pipes = f.by_type("IfcPipeSegment")
    fittings = f.by_type("IfcPipeFitting")
    supports = f.by_type("IfcMechanicalFastener")
    
    print("IFC Entity counts:")
    print(f"  IfcPipeSegment (Straights): {len(pipes)}")
    print(f"  IfcPipeFitting (Bends): {len(fittings)}")
    print(f"  IfcMechanicalFastener (Supports): {len(supports)}\n")
    
    # Inspect custom support reaction forces property set (Pset_TubaSupportForces)
    print("Exported Support Reaction Forces:")
    for sup in supports:
        print(f"  Support Name: {sup.Name}")
        for definition in sup.IsDefinedBy:
            if definition.is_a("IfcRelDefinesByProperties"):
                pset = definition.RelatingPropertyDefinition
                if pset.is_a("IfcPropertySet") and pset.Name == "Pset_TubaSupportForces":
                    for prop in pset.HasProperties:
                        print(f"    {prop.Name}: {prop.NominalValue.wrappedValue}")
except ImportError:
    print("ifcopenshell is not installed in the current environment. Skip property checks.")

IFC Entity counts:
  IfcPipeSegment (Straights): 2
  IfcPipeFitting (Bends): 1
  IfcMechanicalFastener (Supports): 2

Exported Support Reaction Forces:
  Support Name: Support_N0_0
    VerticalReaction_N: 1476.64
    LateralReaction_N: -1.8911e-09
    AxialReaction_N: 2361.33
    TorsionalMoment_Nm: 5.57874e-10
    SupportType: anchor
    FrictionCoefficient: 0.0
  Support Name: Support_N3_1
    VerticalReaction_N: -772.251
    LateralReaction_N: -2.18938e-09
    AxialReaction_N: -2361.33
    TorsionalMoment_Nm: 3.39293e-09
    SupportType: anchor
    FrictionCoefficient: 0.0


## 5. IFC Re-Import

We can read an IFC model back into a standard `TubaModel` using the `IfcImporter` adapter.

In [8]:
importer = IfcImporter()
imported_model = importer.import_model(ifc_filepath)

print(f"Imported Model Summary:")
print(f"  Nodes count: {len(imported_model.nodes)}")
print(f"  Elements count: {len(imported_model.elements)}")
print(f"  Supports count: {len(imported_model.supports)}\n")

# Print nodes in imported model to check geometry resolution
for nid, node in imported_model.nodes.items():
    print(f"  Imported Node {nid}: coords = {node.coords}")

Imported Model Summary:
  Nodes count: 4
  Elements count: 3
  Supports count: 2

  Imported Node N0: coords = [0. 0. 0.]
  Imported Node N1: coords = [3. 0. 0.]
  Imported Node N2: coords = [3.2 0.2 0. ]
  Imported Node N3: coords = [3.2 2.2 0. ]


## Key Takeaways

- Tuba JSON is the canonical, schema-friendly source of truth for agent and API workflows.
- IFC4 export maps pipes, bends, supports, placements, and solver properties into BIM objects.
- Solver properties in IFC are Code_Aster-backed through `load_or_run_code_aster_results(...)`.
- Re-import checks whether external exchange preserved usable geometry and metadata.

Next: `08_expansion_aware_autorouting.ipynb` handles hot-line routing where thermal movement reserves space.
